# Ablation: RoBERTa + Hashtag Prior (No CoT)

**Purpose:** Isolate the contribution of the CoT compression pipeline in RDS by testing whether
RoBERTa + the Symbolic Hashtag Prior alone can match full RDS performance.

**What is stripped vs. kept:**
- ❌ Removed: Qwen CoT generation, Entropy Budgeter, Gradient-Sensitized Local Guardian, LLMLingua-2 compression
- ✅ Kept: RoBERTa classifier (`cardiffnlp/twitter-roberta-base-irony`)
- ✅ Kept: Symbolic Prior (`detect_explicit_irony_signal`) — exact same regexes, same weights
- ✅ Kept: Same dataset split (first 50 = validation, remaining 734 = test)
- ✅ Kept: Same fusion injection logic and numerical parameters from RDS

**Fusion in this ablation:**
Since there is no CoT signal, the base prediction is purely RoBERTa. The hashtag prior is then
injected using the same blending weights as RDS (ωstrong=0.60, ωweak=0.75, αcap=0.30).
When no signal fires, the prediction defaults to RoBERTa alone.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets torch

In [ ]:
# ── Cell 2: Imports and device setup ─────────────────────────────────────────
import torch
import torch.nn.functional as F
import re
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# ── Cell 3: Load dataset — identical split to RDS ────────────────────────────
# First 50 tweets = validation (used for hyperparameter tuning in RDS).
# Tweets 51–784 (index 50–783) = held-out test set → N=734.
# This exactly mirrors the RDS evaluation protocol.

print("Loading TweetEval Irony dataset...")
dataset   = load_dataset("tweet_eval", "irony")
test_data = dataset['test']

VAL_SIZE  = 50
val_data  = test_data.select(range(VAL_SIZE))
eval_data = test_data.select(range(VAL_SIZE, len(test_data)))  # N=734

print(f"Validation split : {len(val_data)} tweets  (indices 0–{VAL_SIZE-1})")
print(f"Test split       : {len(eval_data)} tweets  (indices {VAL_SIZE}–{len(test_data)-1})")

In [ ]:
# ── Cell 4: Load RoBERTa classifier — identical to RDS ───────────────────────
print("Loading RoBERTa classifier...")
roberta_name  = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model     = AutoModelForSequenceClassification.from_pretrained(roberta_name).to(device)
rob_model.eval()
print("RoBERTa loaded.")

In [ ]:
# ── Cell 5: Symbolic Prior — exact copy from RDS (Cell 9/11 of original) ─────
# No changes to regexes, weights, or logic. This is the identical module.

# Definitive author-labeled irony markers
_STRONG_IRONY_RE = re.compile(
    r"#(not|sarcasm|sarcastic|irony|ironic|jk|justkidding|kms|killme|"
    r"killusslow|obviously|surenot|yeahright|fml|eyeroll)\b",
    re.IGNORECASE
)

# Softer hashtag signals
_WEAK_IRONY_RE = re.compile(
    r"#(humor|lol|smh|seriously|really|wow|sure|totally|great|wonderful)\b",
    re.IGNORECASE
)

# Positive emoji immediately followed by a negative emoji → contrast
_EMOJI_CONTRAST_RE = re.compile(
    r"[\U0001F60A\U0001F604\U0001F600\U0001F44D\U00002764\U0001F642\U0001F601]"
    r"\s{0,3}[|,\s]*\s{0,3}"
    r"[\U0001F612\U0001F624\U0001F621\U0001F61E\U0001F611\U0001F644\U0001F614\U0001F622]"
)

# Letter elongation: "Loooove", "Soooo" (3+ repeated chars)
_ELONGATION_RE = re.compile(r"([a-zA-Z])\1{2,}")


def detect_explicit_irony_signal(tweet_text: str) -> float:
    """
    Exact replica of the symbolic prior from RDS.

    Returns:
      0.88  → Strong signal (#not, #sarcasm, etc.)   [ρ_strong]
      0–0.70 → Weak composite signal                 [ρ_cap = 0.70]
      0.0   → No explicit signal

    Parameter values (identical to RDS paper Section 4.3):
      ρ_strong = 0.88
      ρ_weak   = 0.15
      ρ_emoji  = 0.25
      ρ_elong  = 0.12
      ρ_cap    = 0.70
      η        = 3  (min repeated chars for elongation)
    """
    if _STRONG_IRONY_RE.search(tweet_text):
        return 0.88  # ρ_strong

    score = 0.0
    if _WEAK_IRONY_RE.search(tweet_text):
        score += 0.15  # ρ_weak
    if _EMOJI_CONTRAST_RE.search(tweet_text):
        score += 0.25  # ρ_emoji
    if _ELONGATION_RE.search(tweet_text):
        score += 0.12  # ρ_elong

    return min(score, 0.70)  # ρ_cap


print("Symbolic prior module loaded (identical to RDS).")

In [ ]:
# ── Cell 6: RoBERTa classifier helper ────────────────────────────────────────
# Identical to classify_tweet() in RDS.

def classify_tweet(tweet_text: str):
    """Returns (predicted_class, p_ironic) from RoBERTa."""
    inputs = rob_tokenizer(
        tweet_text, return_tensors="pt",
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = rob_model(**inputs)
    probs      = F.softmax(outputs.logits, dim=-1).squeeze().cpu()
    p_ironic   = probs[1].item()
    prediction = int(p_ironic >= 0.5)
    return prediction, p_ironic


print("RoBERTa helper loaded.")

In [ ]:
# ── Cell 7: Ablation fusion — RoBERTa + Hashtag Prior, no CoT ────────────────
#
# Fusion logic mirrors RDS Phase 3 (Prior Injection, Section 3.7) exactly:
#   Strong signal  → α ← min(α, α_cap=0.30), p_fused = ω_strong*p_base + (1-ω_strong)*prior
#   Weak signal    → p_fused = ω_weak*p_base + (1-ω_weak)*prior
#   No signal      → p_fused = p_roberta  (pure RoBERTa, no CoT to blend)
#
# The only structural difference from full RDS:
#   - p_base is always p_roberta (there is no p_cot to blend in)
#   - α is fixed at 1.0 for the base fusion (100% RoBERTa weight)
#   - Entropy budgeter and gradient guardian are not run
#
# Numerical parameters (identical to RDS Section 4.3):
#   ω_strong = 0.60,  α_cap = 0.30,  ω_weak = 0.75

OMEGA_STRONG = 0.60   # blend weight for p_base under strong prior (RDS: ωstrong)
ALPHA_CAP    = 0.30   # max RoBERTa trust when strong prior fires  (RDS: αcap)
OMEGA_WEAK   = 0.75   # blend weight for p_fused under weak prior  (RDS: ωweak)


def ablation_classify(tweet_text: str):
    """
    RoBERTa + Hashtag Prior classifier (no CoT, no entropy, no compression).

    Returns:
        prediction    : int   — 0 (non-ironic) or 1 (ironic)
        p_roberta     : float — raw RoBERTa ironic probability
        hashtag_prior : float — symbolic prior score
        p_fused       : float — final fused probability
    """
    _, p_roberta     = classify_tweet(tweet_text)
    hashtag_prior    = detect_explicit_irony_signal(tweet_text)

    # Base: RoBERTa only (α = 1.0, no CoT)
    p_base = p_roberta

    if hashtag_prior >= 0.88:  # strong signal threshold = ρ_strong
        # Cap RoBERTa trust, recompute base with capped alpha, then blend prior
        # In RDS the cap reduces RoBERTa weight to make room for CoT.
        # Here CoT = 0, so effectively: p_base_capped = ALPHA_CAP * p_roberta + (1-ALPHA_CAP)*0.5
        # But to stay maximally faithful to RDS intent, we treat p_cot = 0.5 (neutral)
        # when CoT is absent — this is the most conservative honest assumption.
        p_cot_neutral = 0.5
        p_base_capped = ALPHA_CAP * p_roberta + (1 - ALPHA_CAP) * p_cot_neutral
        p_fused       = OMEGA_STRONG * p_base_capped + (1 - OMEGA_STRONG) * hashtag_prior

    elif hashtag_prior > 0.0:  # weak signal
        p_fused = OMEGA_WEAK * p_base + (1 - OMEGA_WEAK) * hashtag_prior

    else:  # no signal — pure RoBERTa
        p_fused = p_base

    prediction = int(p_fused >= 0.5)
    return prediction, p_roberta, hashtag_prior, p_fused


print("Ablation fusion function loaded.")

In [ ]:
# ── Cell 8: Run on validation split (first 50) ───────────────────────────────
# Mirrors the RDS validation protocol. Useful for sanity-checking but
# no hyperparameters are tuned here — we use RDS values as-is.

print("Running ablation on VALIDATION split (N=50)...\n")

val_correct = 0
for i in range(len(val_data)):
    tweet      = val_data[i]["text"]
    true_label = val_data[i]["label"]
    pred, p_rob, htag, p_fused = ablation_classify(tweet)
    if pred == true_label:
        val_correct += 1

val_acc = val_correct / len(val_data) * 100
print(f"Validation Accuracy: {val_acc:.1f}%  ({val_correct}/{len(val_data)})")
print("(RDS validation accuracy was 88.0% — difference here isolates CoT contribution on val set)")

In [ ]:
# ── Cell 9: Main evaluation loop — held-out test set (N=734) ─────────────────

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

print(f"Running ablation on TEST split (N={len(eval_data)})...\n")
print(f"{'='*70}")

results_list        = []
correct_predictions = 0

for i in range(len(eval_data)):
    tweet      = eval_data[i]["text"]
    true_label = eval_data[i]["label"]

    pred, p_rob, htag, p_fused = ablation_classify(tweet)

    is_correct = (pred == true_label)
    if is_correct:
        correct_predictions += 1

    total_so_far = i + 1
    running_acc  = correct_predictions / total_so_far * 100
    htag_str     = f" | HTag={htag:.2f}" if htag > 0.0 else ""

    print(
        f"--- Tweet {i+1}: {tweet[:55]}... ---\n"
        f"  RoBERTa={p_rob:.3f}{htag_str} | Fused={p_fused:.3f} "
        f"| Pred={pred} | True={true_label} "
        f"| {'✅' if is_correct else '❌'} "
        f" [{correct_predictions}/{total_so_far} = {running_acc:.1f}%]"
    )

    results_list.append({
        "Tweet"            : tweet[:40] + "...",
        "True"             : true_label,
        "Pred"             : pred,
        "Correct"          : "✅" if is_correct else "❌",
        "RoBERTa P(ironic)": round(p_rob,   3),
        "HTag Prior"       : round(htag,    3),
        "Fused P(ironic)"  : round(p_fused, 3),
    })

print(f"\n{'='*70}")

In [ ]:
# ── Cell 10: Metrics — mirrors Table 1 of the RDS paper ──────────────────────

y_true = [r["True"] for r in results_list]
y_pred = [r["Pred"] for r in results_list]

accuracy   = accuracy_score(y_true, y_pred)
macro_f1   = f1_score(y_true, y_pred, average="macro")
ironic_f1  = f1_score(y_true, y_pred, pos_label=1, average="binary")
ironic_p   = precision_score(y_true, y_pred, pos_label=1, average="binary")
ironic_r   = recall_score(y_true, y_pred, pos_label=1, average="binary")
nonironic_f1 = f1_score(y_true, y_pred, pos_label=0, average="binary")
cm         = confusion_matrix(y_true, y_pred)

tn, fp, fn, tp = cm.ravel()

print("\n" + "="*70)
print(" ABLATION RESULTS: RoBERTa + Hashtag Prior  (No CoT)")
print(" Test set N=734  |  same split as RDS paper")
print("="*70)
print(f"  Accuracy        : {accuracy*100:.1f}%")
print(f"  Macro F1        : {macro_f1:.3f}")
print(f"  Ironic F1       : {ironic_f1:.3f}")
print(f"  Ironic Precision: {ironic_p:.3f}")
print(f"  Ironic Recall   : {ironic_r:.3f}")
print(f"  Non-Ironic F1   : {nonironic_f1:.3f}")
print(f"  Confusion Matrix: TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print("="*70)

print("\n--- For comparison (from RDS paper Table 1) ---")
print("  Method                  Acc    MacroF1  IronicF1  IronicPrec  IronicRec  Non-IronicF1")
print("  RoBERTa-Only Baseline  73.2%   0.706    0.620     0.700       0.557      0.792")
print("  Full RDS Framework     78.1%   0.777    0.747     0.684       0.824      0.806")
print(f"  RoBERTa + Hashtag     {accuracy*100:.1f}%   {macro_f1:.3f}    {ironic_f1:.3f}     {ironic_p:.3f}       {ironic_r:.3f}      {nonironic_f1:.3f}")
print("="*70)

In [ ]:
# ── Cell 11: McNemar's test vs. RoBERTa-Only baseline ────────────────────────
# Mirrors the statistical validation in RDS Section 5.2.
# Run this after also running the RoBERTa-Only baseline notebook to get
# its per-tweet predictions. If you have them, paste them into roberta_only_preds.
#
# If you don't have them yet, skip this cell — it requires the baseline predictions.

from statsmodels.stats.contingency_tables import mcnemar

# --- Paste RoBERTa-Only per-tweet predictions here (list of 0/1, length 734) ---
# roberta_only_preds = [...]  # fill this in

# Uncomment below once roberta_only_preds is available:
# y_ablation = y_pred
# both_wrong = sum(1 for a,b,t in zip(roberta_only_preds, y_ablation, y_true) if a!=t and b!=t)
# ablation_right_base_wrong = sum(1 for a,b,t in zip(roberta_only_preds, y_ablation, y_true) if a!=t and b==t)
# ablation_wrong_base_right = sum(1 for a,b,t in zip(roberta_only_preds, y_ablation, y_true) if a==t and b!=t)
# both_right = sum(1 for a,b,t in zip(roberta_only_preds, y_ablation, y_true) if a==t and b==t)
# table = [[both_right, ablation_wrong_base_right],
#          [ablation_right_base_wrong, both_wrong]]
# result = mcnemar(table, exact=False, correction=True)
# print(f"McNemar chi2={result.statistic:.3f}, p={result.pvalue:.4f}")

print("McNemar cell ready. Fill in roberta_only_preds list to run.")

In [ ]:
# ── Cell 12: Subpopulation breakdown — mirrors Table 2 of RDS paper ──────────
# Breaks down performance on hashtag-triggered vs non-triggered subsets.

triggered_results     = [r for r in results_list if r["HTag Prior"] > 0.0]
non_triggered_results = [r for r in results_list if r["HTag Prior"] == 0.0]

def subset_metrics(subset):
    yt = [r["True"] for r in subset]
    yp = [r["Pred"] for r in subset]
    if len(set(yt)) < 2:  # guard against single-class subset
        return None
    return {
        "n"           : len(subset),
        "accuracy"    : accuracy_score(yt, yp) * 100,
        "macro_f1"    : f1_score(yt, yp, average="macro"),
        "nonironic_f1": f1_score(yt, yp, pos_label=0, average="binary"),
        "ironic_f1"   : f1_score(yt, yp, pos_label=1, average="binary"),
    }

trig_m  = subset_metrics(triggered_results)
notrig_m = subset_metrics(non_triggered_results)

print("\n" + "="*70)
print(" SUBPOPULATION ANALYSIS (mirrors RDS Table 2)")
print("="*70)
print(f"{'Subset':<22} {'n':>5} {'Accuracy':>10} {'Macro F1':>10} {'Non-Ironic F1':>14} {'Ironic F1':>10}")
print("-"*70)
if trig_m:
    print(f"{'Hashtag-Triggered':<22} {trig_m['n']:>5} {trig_m['accuracy']:>9.1f}% {trig_m['macro_f1']:>10.3f} {trig_m['nonironic_f1']:>14.3f} {trig_m['ironic_f1']:>10.3f}")
if notrig_m:
    print(f"{'Non-Triggered':<22} {notrig_m['n']:>5} {notrig_m['accuracy']:>9.1f}% {notrig_m['macro_f1']:>10.3f} {notrig_m['nonironic_f1']:>14.3f} {notrig_m['ironic_f1']:>10.3f}")
print("="*70)

htag_strong = sum(1 for r in results_list if r["HTag Prior"] >= 0.88)
htag_weak   = sum(1 for r in results_list if 0.0 < r["HTag Prior"] < 0.88)
print(f"\nHashtag detector: {len(triggered_results)} fired ({htag_strong} strong, {htag_weak} weak) | {len(non_triggered_results)} silent")

In [ ]:
# ── Cell 13: Full results dataframe ──────────────────────────────────────────

df = pd.DataFrame(results_list)
print(f"Full results table (N={len(df)}):")
display(df)

## Interpretation Guide

Compare these results against Table 1 and Table 2 of the RDS paper:

| Scenario | Implication |
|---|---|
| **Ablation ≈ Full RDS** | CoT compression adds nothing. The contribution claim needs to be dropped or reframed. Paper becomes a neuro-symbolic gating study. |
| **Full RDS > Ablation** | CoT machinery is doing real work. The paper's contribution holds. Difference magnitude determines how much it matters. |
| **Ablation > Full RDS** | CoT compression is actively hurting performance. Significant reframing required. |

Pay particular attention to the **non-triggered subset** (no hashtag). If both ablation and RDS show Ironic F1 ≈ 0 there, that confirms the hashtag prior is the sole driver of irony detection regardless of whether CoT is present.